In [12]:
# Step 1: Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Step 2: Load dataset
df = pd.read_csv('/content/electric_vehicles_spec_2025.csv.csv')
df.head()
# Step 3: Data Cleaning & Feature Selection

# Check for missing values
df.isnull().sum()

# Drop columns that aren't useful for prediction
df = df.drop(['model', 'source_url', 'brand', 'car_body_type', 'segment', 'battery_type'], axis=1)

# Drop rows with missing numeric values (optional for now)
df = df.dropna()

# Confirm shape after cleaning
print("Dataset shape after cleaning:", df.shape)

# Display first few rows again
df.head()
# ✅ FINAL STEP 4 — Safe cleaning + encoding + regression model

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Step 4.1: Make a copy for safety
data = df.copy()

# Step 4.2: Handle specific bad entries like 'Banana Boxes'
data = data.applymap(lambda x: str(x).replace('Banana Boxes', '') if isinstance(x, str) else x)

# Step 4.3: Convert numeric-like columns safely
for col in data.columns:
    # If mostly numbers, try converting
    if data[col].apply(lambda x: str(x).replace('.', '').replace('-', '').isdigit()).sum() > len(data) * 0.5:
        data[col] = pd.to_numeric(data[col], errors='coerce')

# Step 4.4: Encode non-numeric columns (e.g., drivetrain)
for cat_col in data.select_dtypes(include=['object']).columns:
    data[cat_col] = data[cat_col].astype('category').cat.codes

# Step 4.5: Drop NaNs
data = data.dropna()

print("✅ Cleaned dataset shape:", data.shape)

# Step 4.6: Define features and target
X = data.drop('range_km', axis=1)
y = data['range_km']

# Step 4.7: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4.8: Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 4.9: Evaluate model
y_pred = model.predict(X_test)

print("✅ Model trained successfully!")
print("R² Score:", round(r2_score(y_test, y_pred), 3))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("MSE:", round(mean_squared_error(y_test, y_pred), 3))
# 🚗 Step 5: EVA Chatbot — Talk to Your Model 💬

def eva_chat():
    print("🔋 EVA: Your Electric Vehicle Range Assistant ⚡")
    print("Type 'exit' to end the chat.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("EVA: See you later! Keep charging forward 🔋💨")
            break

        # Try to extract battery capacity (kWh) from input
        import re
        match = re.search(r'(\d+\.?\d*)', user_input)

        if match:
            battery_capacity = float(match.group(1))
            # Use a simple linear correlation between battery and range
            # We'll use model coefficients to estimate
            try:
                # Pick a sample input row (average of all features)
                sample = X.mean().to_frame().T
                sample['battery_capacity_kWh'] = battery_capacity

                # Predict range
                pred = model.predict(sample)[0]
                print(f"EVA: Estimated range ≈ {pred:.2f} km ⚡")
            except Exception as e:
                print("EVA: Hmm, I couldn't compute that. Try rephrasing?")
        else:
            print("EVA: Please mention the battery size in kWh (e.g., 'range for 60 kWh EV').")

# Run the chatbot
eva_chat()


/tmp/ipython-input-1117983137.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(lambda x: str(x).replace('Banana Boxes', '') if isinstance(x, str) else x)


Dataset shape after cleaning: (265, 16)
✅ Cleaned dataset shape: (265, 16)
✅ Model trained successfully!
R² Score: 0.961
MAE: 15.475
MSE: 332.759
🔋 EVA: Your Electric Vehicle Range Assistant ⚡
Type 'exit' to end the chat.

You: predict range for 65 kWh EV
EVA: Estimated range ≈ 355.23 km ⚡


KeyboardInterrupt: Interrupted by user